<a href="https://colab.research.google.com/github/Sam-Gyu/naive-bayes-from-scratch/blob/feature%2Fdata-pipeline/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import urllib.request
import os

# utility functions
def compute_mean(values):
    """Formula: μ = (1/n) * Σ xi"""
    if len(values) == 0:
        return 0
    return sum(values) / len(values)

def compute_variance(values):
    """Formula: σ2 = (1/n) * Σ (xi - μ)2"""
    if len(values) == 0:
        return 0
    mu = compute_mean(values)
    return sum((x - mu) ** 2 for x in values) / len(values)

def compute_accuracy(y_true, y_pred):
    """Formula: Accuracy = correct_predictions / total_samples"""
    if len(y_true) == 0:
        return 0
    correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
    return correct / len(y_true)



In [15]:
def load_abalone_data(path):
    # Check if file exists, if not, download it
    if not os.path.exists(path):
        print(f"File {path} not found. Downloading from UCI repository...")
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
        urllib.request.urlretrieve(url, path)
        print(f"Downloaded {path}")

    # Column names from abalone.names
    columns = ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight',
               'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
    df = pd.read_csv(path, names=columns)

    # Convert Rings → (Young, Adult, Old)
    def categorize_rings(rings):
        if rings <= 8:
            return 'Young'
        elif rings <= 11:  # 9-11 inclusive
            return 'Adult'
        else:
            return 'Old'

    df['Age_Group'] = df['Rings'].apply(categorize_rings)

    # Select only the numeric features (excluding Sex and Rings)
    feature_columns = ['Length', 'Diameter', 'Height', 'Whole_weight',
                       'Shucked_weight', 'Viscera_weight', 'Shell_weight']
    X = df[feature_columns].values
    y = df['Age_Group'].values

    return X, y, df

def train_test_split_manual(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    indices = np.arange(len(X))
    np.random.shuffle(indices)

    test_samples = int(len(X) * test_size)
    test_idx = indices[:test_samples]
    train_idx = indices[test_samples:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [16]:
# Example Usage
if __name__ == "__main__":
    # Load the dataset
    X, y, df_full = load_abalone_data('abalone.data')
    print("Abalone data loaded and Rings converted to Age_Group.")
    print(f"Dataset shape: {X.shape}")
    print(f"Feature columns: Length, Diameter, Height, Whole_weight, Shucked_weight, Viscera_weight, Shell_weight")

    # Show class distribution
    class_counts = pd.Series(y).value_counts()
    print(f"\nClass distribution:")
    for age_group, count in class_counts.items():
        print(f"  {age_group}: {count} samples ({count/len(y)*100:.1f}%)")

    # Perform train/test split
    X_train, X_test, y_train, y_test = train_test_split_manual(X, y)
    print(f"\nData split into training ({len(X_train)} samples) and testing ({len(X_test)} samples).")

    # Example of using compute_mean and compute_variance
    sample_values = [1, 2, 3, 4, 5]
    mean_val = compute_mean(sample_values)
    variance_val = compute_variance(sample_values)
    print(f"\nSample values: {sample_values}")
    print(f"Computed Mean: {mean_val}")
    print(f"Computed Variance: {variance_val}")

    # Example of using compute_accuracy (dummy prediction)
    dummy_y_pred = y_test.copy()
    # Introduce some errors for demonstration
    if len(dummy_y_pred) > 0:
        dummy_y_pred[0] = 'Wrong'
    if len(dummy_y_pred) > 1:
        dummy_y_pred[1] = 'Also Wrong'

    accuracy_val = compute_accuracy(y_test, dummy_y_pred)
    print(f"\nDummy Accuracy: {accuracy_val:.4f}")

Abalone data loaded and Rings converted to Age_Group.
Dataset shape: (4177, 7)
Feature columns: Length, Diameter, Height, Whole_weight, Shucked_weight, Viscera_weight, Shell_weight

Class distribution:
  Adult: 1810 samples (43.3%)
  Young: 1407 samples (33.7%)
  Old: 960 samples (23.0%)

Data split into training (3342 samples) and testing (835 samples).

Sample values: [1, 2, 3, 4, 5]
Computed Mean: 3.0
Computed Variance: 2.0

Dummy Accuracy: 0.9976


In [17]:

    # Show feature statistics per class (helpful for Ahmed's part)
    print("\n" + "="*60)
    print("Feature Statistics per Class (for Ahmed's Gaussian NB):")
    print("="*60)

    # Create a dataframe with training features and labels for easy grouping
    X_train_df = pd.DataFrame(X_train, columns=['Length', 'Diameter', 'Height', 'Whole_weight',
                                                 'Shucked_weight', 'Viscera_weight', 'Shell_weight'])
    X_train_df['Age_Group'] = y_train

    for age_group in ['Young', 'Adult', 'Old']:
        subset = X_train_df[X_train_df['Age_Group'] == age_group]
        print(f"\n{age_group} Class (n={len(subset)}):")
        for feature in ['Length', 'Diameter', 'Height', 'Whole_weight',
                       'Shucked_weight', 'Viscera_weight', 'Shell_weight']:
            mean_val = compute_mean(subset[feature].values)
            var_val = compute_variance(subset[feature].values)
            print(f"  {feature}: mean={mean_val:.4f}, variance={var_val:.4f}")



Feature Statistics per Class (for Ahmed's Gaussian NB):

Young Class (n=1129):
  Length: mean=0.4229, variance=0.0120
  Diameter: mean=0.3226, variance=0.0080
  Height: mean=0.1070, variance=0.0019
  Whole_weight: mean=0.4354, variance=0.0932
  Shucked_weight: mean=0.1991, variance=0.0215
  Viscera_weight: mean=0.0938, variance=0.0046
  Shell_weight: mean=0.1222, variance=0.0066

Adult Class (n=1436):
  Length: mean=0.5716, variance=0.0076
  Diameter: mean=0.4465, variance=0.0051
  Height: mean=0.1518, variance=0.0009
  Whole_weight: mean=0.9875, variance=0.1844
  Shucked_weight: mean=0.4415, variance=0.0441
  Viscera_weight: mean=0.2179, variance=0.0096
  Shell_weight: mean=0.2759, variance=0.0121

Old Class (n=777):
  Length: mean=0.5868, variance=0.0066
  Diameter: mean=0.4630, variance=0.0045
  Height: mean=0.1650, variance=0.0009
  Whole_weight: mean=1.1151, variance=0.2097
  Shucked_weight: mean=0.4452, variance=0.0440
  Viscera_weight: mean=0.2385, variance=0.0101
  Shell_weigh

In [18]:
    # Verify the split and data integrity
    print("\n" + "="*60)
    print("Data Verification:")
    print("="*60)
    print(f"Training set - X shape: {X_train.shape}, y shape: {y_train.shape}")
    print(f"Test set - X shape: {X_test.shape}, y shape: {y_test.shape}")
    print(f"\nSample from training set (first 3 samples):")
    for i in range(min(3, len(X_train))):
        print(f"  Sample {i+1}: {X_train[i]}")
        print(f"  Label: {y_train[i]}")


Data Verification:
Training set - X shape: (3342, 7), y shape: (3342,)
Test set - X shape: (835, 7), y shape: (835,)

Sample from training set (first 3 samples):
  Sample 1: [0.365  0.275  0.085  0.223  0.098  0.0375 0.075 ]
  Label: Young
  Sample 2: [0.55   0.445  0.125  0.672  0.288  0.1365 0.21  ]
  Label: Adult
  Sample 3: [0.475  0.355  0.1    0.5035 0.2535 0.091  0.14  ]
  Label: Young
